# One-Shot Foundation Classifier: Version 1
**ITCS 6169 Assignment 3: Foundation Model**

*By Tahiyat Chy*


## Initial Setup

In [1]:
!pip install -U -q open_clip_torch timm # ensure latest version installed

In [10]:
# Core imports
from pathlib import Path
import random
import json
import copy
import heapq

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torchvision.transforms as T

import open_clip

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.14.1+cu130
CUDA available: True


In [3]:
# Mount Google Drive (when running in Colab)
try:
    from google.colab import drive
    drive.mount("/content/gdrive")
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")

Mounted at /content/gdrive


In [4]:
# Path set to assignment directory
PROJECT_ROOT = Path("/content/gdrive/MyDrive/ITCS_6169/assignment3")
DATA_ROOT = PROJECT_ROOT / "data"
TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT = DATA_ROOT / "test"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Training data:", TRAIN_ROOT)
print("Testing data:", TEST_ROOT)
print("Checkpoints dir:", CHECKPOINT_DIR)

Project root: /content/gdrive/MyDrive/ITCS_6169/assignment3
Training data: /content/gdrive/MyDrive/ITCS_6169/assignment3/data/train
Testing data: /content/gdrive/MyDrive/ITCS_6169/assignment3/data/test
Checkpoints dir: /content/gdrive/MyDrive/ITCS_6169/assignment3/checkpoints


In [5]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## One-Shot Split & Dataloaders

In [6]:
BATCH_SIZE = 64
NUM_CLASSES = 16
SHOTS_PER_CLASS = 1  # Strictly 1 sample per class


class IndexedImageDataset(Dataset):
    """Loads images from (path, label) pairs using a given transform."""

    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        with Image.open(path) as img:
            img = img.convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.samples)


def build_one_shot_splits(root_dir, shots_per_class=1, seed=0):
    """
    Selects strictly `shots_per_class` (default=1) image per class for support/training,
    and places remaining images into a held-out validation pool.
    """
    rng = random.Random(seed)
    base_ds = datasets.ImageFolder(root=root_dir)
    class_names = base_ds.classes

    by_class = {c_idx: [] for c_idx in range(len(class_names))}
    for path, label in base_ds.samples:
        by_class[label].append((path, label))

    support_samples = []
    val_samples = []

    for c_idx, items in by_class.items():
        if len(items) < shots_per_class:
            raise ValueError(
                f"Class {class_names[c_idx]} has {len(items)} images (< {shots_per_class})."
            )
        shuffled = items.copy()
        rng.shuffle(shuffled)
        support_samples.extend(shuffled[:shots_per_class])
        val_samples.extend(shuffled[shots_per_class:])

    # Sort support samples by class index
    support_samples.sort(key=lambda x: x[1])
    return support_samples, val_samples, class_names

    VIEWS_PER_SHOT = 16  # Number of augmented views synthesized per single support image


class MultiViewSupportDataset(Dataset):
    """
    Generates `views_per_shot` augmented views strictly from the 1-shot support samples
    (16 base images -> 16 * views_per_shot augmented keys) without touching any extra training data.
    """

    def __init__(self, support_samples, img_size=384, views_per_shot=16, seed=0):
        self.support_samples = support_samples
        self.views_per_shot = views_per_shot
        self.seed = seed

        # CLIP/SigLIP normalization stats (mean=0.5, std=0.5 for SigLIP)
        self.aug_transform = T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.65, 1.0), ratio=(0.85, 1.15), interpolation=T.InterpolationMode.BICUBIC),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
            T.ToTensor(),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.support_samples) * self.views_per_shot

    def __getitem__(self, idx):
        sample_idx = idx // self.views_per_shot
        view_idx = idx % self.views_per_shot
        path, label = self.support_samples[sample_idx]

        # Deterministic per-view seed for reproducibility
        torch.manual_seed(self.seed + idx)
        random.seed(self.seed + idx)

        with Image.open(path) as img:
            img = img.convert("RGB")
        tensor = self.aug_transform(img)
        return tensor, label

## Model & Prototype

In [7]:
RICH_PROMPT_TEMPLATES = [
    "a photo of a {}.",
    "a clear photograph of a {}.",
    "a close-up photo of a {}.",
    "a high-resolution photo of a {}.",
    "a centered photo of a {}.",
    "a bright photo of a {}.",
    "a photo of the {}.",
    "a cropped photo of a {}.",
    "a clean image showing a {}.",
    "an image of {} in natural lighting.",
    "a detailed view of a {}.",
    "a photograph focusing on a {}.",
]

VIEWS_PER_SHOT = 16

class MultiViewTipAdapterF(nn.Module):
    """
    Tip-Adapter-F with multi-view 1-shot support keys, L2 anchor regularization,
    and 2-view test-time flip averaging.
    """

    def __init__(
        self,
        clip_model,
        tokenizer,
        class_names,
        support_features,
        support_labels,
        templates=RICH_PROMPT_TEMPLATES,
        alpha=2.5,
        beta=4.5,
        logit_scale=25.0,
    ):
        super().__init__()
        self.clip_model = clip_model
        self.tokenizer = tokenizer
        self.class_names = class_names
        self.alpha = alpha
        self.beta = beta
        self.logit_scale = logit_scale

        # Freeze foundation backbone weights
        for p in self.clip_model.parameters():
            p.requires_grad = False

        # 1. Text prototype embeddings averaged across 12 templates: (C, D)
        text_embeds = self._build_text_embeddings(class_names, templates)
        self.register_buffer("text_features", text_embeds)

        # 2. Multi-view visual cache keys (N_cache, D) & one-hot values (N_cache, C)
        init_keys = F.normalize(support_features.detach(), dim=-1)
        num_cache, feat_dim = init_keys.shape

        self.register_buffer("init_cache_keys", init_keys.clone())
        one_hot = F.one_hot(support_labels, num_classes=len(class_names)).float()
        self.register_buffer("cache_values", one_hot)

        # Learnable cache keys (Tip-Adapter-F) initialized with multi-view support embeddings
        self.cache_keys = nn.Linear(feat_dim, num_cache, bias=False)
        with torch.no_grad():
            self.cache_keys.weight.copy_(init_keys)

    @torch.no_grad()
    def _build_text_embeddings(self, class_names, templates):
        device = next(self.clip_model.parameters()).device
        per_class_feats = []
        for raw_name in class_names:
            readable_name = raw_name.replace("_", " ").replace("-", " ")
            prompts = [t.format(readable_name) for t in templates]
            tokens = self.tokenizer(prompts).to(device)
            feats = self.clip_model.encode_text(tokens)
            feats = F.normalize(feats, dim=-1)
            mean_feat = F.normalize(feats.mean(dim=0, keepdim=True), dim=-1)
            per_class_feats.append(mean_feat)
        return torch.cat(per_class_feats, dim=0)  # (C, D)

    @torch.no_grad()
    def encode_images(self, images, use_flip_tta=True):
        feats = F.normalize(self.clip_model.encode_image(images), dim=-1)
        if use_flip_tta:
            flipped = torch.flip(images, dims=[-1])
            feats_flip = F.normalize(self.clip_model.encode_image(flipped), dim=-1)
            feats = F.normalize(0.5 * (feats + feats_flip), dim=-1)
        return feats

    def logits_from_features(self, img_features):
        img_features = F.normalize(img_features, dim=-1)

        # 1. Zero-shot text similarity logits
        text_logits = self.logit_scale * (img_features @ self.text_features.T)

        # 2. Fine-tuned multi-view cache similarity logits
        norm_keys = F.normalize(self.cache_keys.weight, dim=-1)  # (N_cache, D)
        affinity = img_features @ norm_keys.T                    # (B, N_cache)
        cache_weights = torch.exp(-self.beta * (1.0 - affinity))
        cache_logits = cache_weights @ self.cache_values         # (B, C)

        return text_logits + self.alpha * cache_logits

    def anchor_loss(self):
        """Prevents fine-tuned cache keys from drifting too far from initial 1-shot embeddings."""
        norm_keys = F.normalize(self.cache_keys.weight, dim=-1)
        return F.mse_loss(norm_keys, self.init_cache_keys)

    def forward(self, images, use_flip_tta=True):
        with torch.no_grad():
            img_features = self.encode_images(images, use_flip_tta=use_flip_tta)
        return self.logits_from_features(img_features)

## Training & Adaptation

In [8]:
# Upgrade to SigLIP-2 SO400M
MODEL_NAME = "hf-hub:timm/ViT-SO400M-16-SigLIP2-384"

clip_model, _, preprocess_val = open_clip.create_model_and_transforms(
    MODEL_NAME, device=device
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
clip_model.eval()

# Infer model input resolution from preprocess_val
IMG_SIZE = preprocess_val.transforms[0].size
if isinstance(IMG_SIZE, (tuple, list)):
    IMG_SIZE = IMG_SIZE[0]

# Build strict 1-shot support set (16 images) and identical 2,384-image val pool
support_samples, val_samples, class_names = build_one_shot_splits(
    TRAIN_ROOT, shots_per_class=SHOTS_PER_CLASS, seed=SEED
)
assert len(support_samples) == NUM_CLASSES * SHOTS_PER_CLASS, "Violated 1-shot rule!"

# Multi-view augmented dataset built strictly from the 16 support images
mv_support_dataset = MultiViewSupportDataset(
    support_samples, img_size=IMG_SIZE, views_per_shot=VIEWS_PER_SHOT, seed=SEED
)
val_dataset = IndexedImageDataset(val_samples, transform=preprocess_val)

mv_support_loader = DataLoader(mv_support_dataset, batch_size=64, shuffle=False, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Base 1-shot images: {len(support_samples)} | Augmented support views: {len(mv_support_dataset)} | Val images: {len(val_dataset)}")

# Extract multi-view support visual embeddings once
all_sup_feats, all_sup_labels = [], []
with torch.no_grad():
    for imgs, labels in mv_support_loader:
        imgs = imgs.to(device)
        feats = F.normalize(clip_model.encode_image(imgs), dim=-1)
        all_sup_feats.append(feats)
        all_sup_labels.append(labels.to(device))

sup_feats = torch.cat(all_sup_feats, dim=0)    # (16 * VIEWS_PER_SHOT, D)
sup_labels = torch.cat(all_sup_labels, dim=0)  # (16 * VIEWS_PER_SHOT,)

model = MultiViewTipAdapterF(
    clip_model=clip_model,
    tokenizer=tokenizer,
    class_names=class_names,
    support_features=sup_feats,
    support_labels=sup_labels,
    alpha=2.5,
    beta=4.5,
    logit_scale=25.0,
).to(device)

open_clip_config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B / 4.54GB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

Base 1-shot images: 16 | Augmented support views: 256 | Val images: 2384


In [11]:
def evaluate_model(model, loader, device, use_flip_tta=True):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images, use_flip_tta=use_flip_tta)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / max(total, 1)


def fit_tip_adapter_f(model, sup_feats, sup_labels, epochs=60, lr=2e-3, anchor_weight=0.5):
    """
    Fine-tunes the multi-view visual cache keys on the 1-shot augmented views
    with cosine LR schedule and L2 anchor regularization.
    """
    optimizer = torch.optim.AdamW([model.cache_keys.weight], lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    best_keys = copy.deepcopy(model.cache_keys.weight.detach())
    best_loss = float("inf")

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model.logits_from_features(sup_feats)
        loss = criterion(logits, sup_labels) + anchor_weight * model.anchor_loss()
        loss.backward()
        optimizer.step()
        scheduler.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_keys = copy.deepcopy(model.cache_keys.weight.detach())

    with torch.no_grad():
        model.cache_keys.weight.copy_(best_keys)
    return best_loss


@torch.no_grad()
def evaluate_with_dynamic_tda(
    model, loader, device, max_cache_per_class=3, entropy_thresh=0.45, gamma=1.5, beta_tda=5.0
):
    """
    Training-Free Dynamic Adapter (TDA):
    Maintains a per-class priority queue of the lowest-entropy test embeddings seen so far.
    High-entropy (uncertain) samples are ignored so model weights never drift.
    """
    model.eval()
    num_classes = len(model.class_names)
    # Per-class min-heaps keyed by (-entropy, tie_breaker, feature_tensor) so pop removes highest-entropy item
    pos_cache = {c: [] for c in range(num_classes)}
    step_counter = 0

    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        feats = model.encode_images(images, use_flip_tta=True)  # (B, D)
        base_logits = model.logits_from_features(feats)          # (B, C)

        probs = F.softmax(base_logits, dim=-1)
        entropies = -(probs * torch.log(probs + 1e-8)).sum(dim=-1)
        pred_classes = probs.argmax(dim=-1)

        # Compute dynamic cache logits from currently accepted high-confidence test samples
        tda_logits = torch.zeros_like(base_logits)
        cached_keys, cached_vals = [], []
        for c, items in pos_cache.items():
            for _, _, f_vec in items:
                cached_keys.append(f_vec)
                one_hot = torch.zeros(num_classes, device=device)
                one_hot[c] = 1.0
                cached_vals.append(one_hot)

        if len(cached_keys) > 0:
            k_mat = torch.stack(cached_keys, dim=0)  # (K, D)
            v_mat = torch.stack(cached_vals, dim=0)  # (K, C)
            aff = feats @ k_mat.T
            weights = torch.exp(-beta_tda * (1.0 - aff))
            tda_logits = weights @ v_mat

        final_logits = base_logits + gamma * tda_logits
        final_preds = final_logits.argmax(dim=1)
        correct += (final_preds == labels).sum().item()
        total += labels.size(0)

        # Update dynamic priority queue with confident predictions from this batch
        for i in range(feats.size(0)):
            ent = float(entropies[i].item())
            if ent < entropy_thresh:
                c = int(pred_classes[i].item())
                heap = pos_cache[c]
                entry = (-ent, step_counter, feats[i].detach())
                step_counter += 1
                if len(heap) < max_cache_per_class:
                    heapq.heappush(heap, entry)
                elif -ent > heap[0][0]:  # Strictly lower entropy than worst cached item
                    heapq.heapreplace(heap, entry)

    return correct / max(total, 1)


# Zero-Shot Baseline (alpha = 0)
orig_alpha = model.alpha
model.alpha = 0.0
zs_acc = evaluate_model(model, val_loader, device, use_flip_tta=True)
print(f"SigLIP-2 Zero-Shot (Text Only + Flip TTA) Val Acc: {zs_acc:.4f}")

# Multi-View Tip-Adapter (Training-Free, before key fine-tuning)
model.alpha = orig_alpha
tf_acc = evaluate_model(model, val_loader, device, use_flip_tta=True)
print(f"1-Shot Multi-View Tip-Adapter (Training-Free) Val Acc: {tf_acc:.4f}")

# Fine-Tuned Multi-View Tip-Adapter-F
best_loss = fit_tip_adapter_f(model, sup_feats, sup_labels, epochs=60, lr=2e-3, anchor_weight=0.5)
ft_acc = evaluate_model(model, val_loader, device, use_flip_tta=True)
print(f"1-Shot Multi-View Tip-Adapter-F Val Acc: {ft_acc:.4f} (Loss: {best_loss:.4f})")

# Tip-Adapter-F + Confidence-Gated Dynamic TDA
tda_acc = evaluate_with_dynamic_tda(model, val_loader, device, max_cache_per_class=3, entropy_thresh=0.45, gamma=1.5)
print(f"1-Shot Tip-Adapter-F + Dynamic TDA Val Acc: {tda_acc:.4f}")

SigLIP-2 Zero-Shot (Text Only + Flip TTA) Val Acc: 0.8859
1-Shot Multi-View Tip-Adapter (Training-Free) Val Acc: 0.8876
1-Shot Multi-View Tip-Adapter-F Val Acc: 0.8729 (Loss: 0.3167)
1-Shot Tip-Adapter-F + Dynamic TDA Val Acc: 0.8658
